# 05 · Controller agent
Documentation: `docs/agents/controller_agent.md`.

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_rows', 80); pd.set_option('display.width', 200)
D = ROOT / 'data'
def ok(p):
    if not Path(p).exists(): print('File not available yet:', p); return False
    return True
from core.config import load_config
cfg = load_config()
dec = json.loads((D/'controller/decision.json').read_text())
print('Date:', dec['date'], '· verdict:', dec['verdict'], '· active portfolio:', dec['active_portfolio'])

## Outcome and margin of each check

In [ ]:
rows = []
for lvl in ['hard','soft']:
    for x in dec[lvl]:
        v, l = x['value'], x['limit']
        margin = (v - l) if isinstance(v, (int, float)) and isinstance(l, (int, float)) else None
        rows.append({'level': lvl, 'check': x['check'], 'value': v, 'limit': l, 'margin': margin, 'ok': x['ok']})
pd.DataFrame(rows)

## Verdict history

In [ ]:
if ok(D/'controller/decisions.csv'): display(pd.read_csv(D/'controller/decisions.csv'))

## What would happen with different thresholds

In [ ]:
rep = json.loads((D/'modeling/risk_report.json').read_text()); mc = rep['monte_carlo'].get('1') or rep['monte_carlo'].get(1)
for s in [-0.12, -0.15, -0.18, -0.22, -0.25]:
    print(f"CVaR 95% threshold {s:+.0%}: {'pass' if mc['CVaR95'] >= s else 'BLOCK'} (value {mc['CVaR95']:+.1%})")
for s in [0.03, 0.05, 0.10, 0.12]:
    print(f"P(DD>20%) threshold {s:.0%}: {'pass' if mc['prob_dd20'] <= s else 'BLOCK'} (value {mc['prob_dd20']:.1%})")

## Current thresholds

In [ ]:
cfg['controller']